# UNIDAD I: Introducción a algoritmos y estructuras de datos
## 🏋️ Capítulo 2 DSAP: Ejercicios - Polimorfismo y ADTs
### Programación III - Lic. en Sistemas

**Capítulo:** 2 — OOP  
**Clase:** 2 de 2 — Herencia, ABCs y Polimorfismo  
**Secciones:** §2.4 Herencia · §2.7 ABCs

---

> Esta clase profundiza en herencia y clases base abstractas. Vas a construir jerarquías y contratos formales.

| Ícono | Dificultad | Tiempo estimado |
|-------|-----------|----------------|
| 🟢 | Básico | 10–15 min |
| 🟡 | Intermedio | 20–30 min |
| 🔴🔴 | Desafío | 40–60 min |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap02/Ejercicios_Clase2_PolimorfismoADTs.ipynb)

## 📋 Resumen de ejercicios

| # | Ejercicio | Tópico principal | Dificultad |
|---|-----------|-----------------|-----------|
| 1 | `Cuadrado(Rectangulo)` | Herencia, `super()`, `__init__` especializado | 🟢 |
| 2 | `FiguraGeometrica` ABC | `abc.ABC`, `@abstractmethod`, métodos concretos | 🟢 |
| 3 | Jerarquía `Progresion` | Template Method, `_avanzar()` gancho | 🟡 |
| 4 | `RangoPersonalizado` | ABC `Secuencia`, `__len__`, `__getitem__` | 🟡 |
| 🏆 | `SecuenciaOrdenada` | ABC + herencia + mixin methods | 🔴🔴 |

**Conceptos necesarios:** Notebooks 2/3 y 3/3 — Herencia, Encapsulamiento, ABCs

## ⚙️ Configuración

In [ ]:
import sys
import math
import unittest
import functools
from abc import ABC, abstractmethod

assert sys.version_info >= (3, 8)
print(f"Python {sys.version}")
print("Listo para los ejercicios de Polimorfismo y ADTs — Cap. 2")

# Reutilizamos Rectangulo y Secuencia del Notebook anterior
import functools

@functools.total_ordering
class Rectangulo:
    def __init__(self, ancho, alto):
        if ancho <= 0 or alto <= 0: raise ValueError("Dimensiones positivas")
        self._ancho = float(ancho)
        self._alto  = float(alto)
    @property
    def ancho(self):    return self._ancho
    @property
    def alto(self):     return self._alto
    @property
    def area(self):     return self._ancho * self._alto
    @property
    def perimetro(self):return 2*(self._ancho + self._alto)
    def es_cuadrado(self): return self._ancho == self._alto
    def __repr__(self): return f"Rectangulo(ancho={self._ancho}, alto={self._alto})"
    def __str__(self):  return f"Rectángulo {self._ancho:.0f}×{self._alto:.0f} (área={self.area:.2f})"
    def __eq__(self, other):
        if not isinstance(other, Rectangulo): return NotImplemented
        return abs(self.area - other.area) < 1e-9
    def __lt__(self, other):
        if not isinstance(other, Rectangulo): return NotImplemented
        return self.area < other.area


class Secuencia(ABC):
    @abstractmethod
    def __len__(self): pass
    @abstractmethod
    def __getitem__(self, j): pass
    def __contains__(self, val):
        return any(self[j] == val for j in range(len(self)))
    def __iter__(self):
        j = 0
        while j < len(self):
            yield self[j]; j += 1
    def index(self, val):
        for j in range(len(self)):
            if self[j] == val: return j
        raise ValueError(f"{val!r} no está en la secuencia")
    def count(self, val):
        return sum(1 for j in range(len(self)) if self[j] == val)

print("Clases auxiliares cargadas: Rectangulo, Secuencia (ABC)")

## Ejercicio 1 — `Cuadrado(Rectangulo)` 🟢

**Concepto:** Herencia simple — especialización de una clase existente.

Un cuadrado **es un** rectángulo donde `ancho == alto`.
Implementá la clase `Cuadrado` que:

1. Hereda de `Rectangulo`.
2. Recibe un único argumento `lado` en `__init__`.
3. Delega a `super().__init__(lado, lado)` — sin duplicar lógica.
4. Sobreescribe `__repr__` y `__str__` para mostrar "Cuadrado" en lugar de "Rectángulo".
5. El método `es_cuadrado()` **heredado** debe devolver `True` siempre.

Este ejercicio ilustra el principio de **sustitución de Liskov**: cualquier código que use
`Rectangulo` debe poder usar `Cuadrado` sin cambios.

| Método | ¿Heredado o sobreescrito? |
|--------|--------------------------|
| `area` | Heredado (property) |
| `perimetro` | Heredado (property) |
| `es_cuadrado()` | Heredado — siempre `True` |
| `__eq__` / `__lt__` | Heredados (`@total_ordering`) |
| `__repr__` / `__str__` | **Sobreescritos** |

In [ ]:
class Cuadrado(Rectangulo):
    """Un rectángulo con todos sus lados iguales."""

    def __init__(self, lado: float):
        if lado <= 0: raise ValueError("El lado debe ser positivo")
        super().__init__(lado, lado)   # delega — no repite validación

    @property
    def lado(self) -> float:
        return self._ancho

    def __repr__(self) -> str:
        return f"Cuadrado(lado={self._ancho})"

    def __str__(self) -> str:
        return f"Cuadrado de lado {self._ancho:.2f} (área={self.area:.2f})"


# ── Demo ──────────────────────────────────────────────────────────────────────
c = Cuadrado(4)
r = Rectangulo(4, 3)

print(repr(c))                          # Cuadrado(lado=4.0)
print(str(c))                           # Cuadrado de lado 4.00 (área=16.00)
print(f"area      = {c.area}")          # 16.0
print(f"perimetro = {c.perimetro}")     # 16.0
print(f"es_cuadrado: {c.es_cuadrado()}")# True

# Principio de sustitución de Liskov
print(f"\nc > r: {c > r}")              # True  (16 > 12)
print(f"c == Cuadrado(4): {c == Cuadrado(4)}")  # True

# isinstance — Cuadrado ES un Rectangulo
print(f"\nisinstance(c, Cuadrado):    {isinstance(c, Cuadrado)}")
print(f"isinstance(c, Rectangulo):  {isinstance(c, Rectangulo)}")
print(f"isinstance(r, Cuadrado):    {isinstance(r, Cuadrado)}")

### 🧪 Tests — Ejercicio 1

In [ ]:
class TestCuadrado(unittest.TestCase):

    def test_area_es_lado_cuadrado(self):
        c = Cuadrado(5)
        self.assertEqual(c.area, 25)

    def test_perimetro_es_cuatro_lados(self):
        self.assertEqual(Cuadrado(3).perimetro, 12)

    def test_es_siempre_cuadrado(self):
        self.assertTrue(Cuadrado(7).es_cuadrado())

    def test_hereda_de_rectangulo(self):
        self.assertIsInstance(Cuadrado(2), Rectangulo)

    def test_repr_contiene_cuadrado(self):
        self.assertIn("Cuadrado", repr(Cuadrado(4)))

    def test_str_contiene_cuadrado(self):
        self.assertIn("Cuadrado", str(Cuadrado(4)))

    def test_comparacion_heredada(self):
        self.assertGreater(Cuadrado(5), Cuadrado(3))
        self.assertEqual(Cuadrado(4), Rectangulo(2, 8))   # misma area

    def test_lado_invalido(self):
        with self.assertRaises((ValueError, TypeError)):
            Cuadrado(-1)

res = unittest.TextTestRunner(verbosity=2).run(
    unittest.TestLoader().loadTestsFromTestCase(TestCuadrado))
print(f"\n{'OK' if res.wasSuccessful() else 'FAILED'} — {res.testsRun} tests")

## Ejercicio 2 — Clase Base Abstracta `FiguraGeometrica` 🟢

**Concepto:** Clases Base Abstractas (ABCs) — interfaz + comportamiento heredado.

Diseñá una jerarquía de figuras geométricas siguiendo el patrón del §2.7 del libro:

```
FiguraGeometrica (ABC)
├── area()          ← abstractmethod
├── perimetro()     ← abstractmethod
└── describir()     ← concreto: devuelve string con nombre, area y perimetro

Circulo(FiguraGeometrica)
TrianguloRectangulo(FiguraGeometrica)
```

**Objetivos:**
- No se puede instanciar `FiguraGeometrica` directamente.
- `Circulo` y `TrianguloRectangulo` deben implementar ambos métodos abstractos.
- El método `describir()` se hereda sin cambios y funciona para todas las figuras.

### Pistas

- Usá `from abc import ABC, abstractmethod` (ya importado en la celda de setup).
- `math.pi` y `math.sqrt` están disponibles.
- El área de un triángulo rectángulo de catetos `a` y `b` es `(a * b) / 2`.
- El perímetro del mismo es `a + b + sqrt(a² + b²)`.
- Para demostrar que la ABC establece un **contrato**: intentá crear `FiguraGeometrica()` — debería lanzar `TypeError`.

**Bonus:** Ordená una lista heterogénea de figuras por área usando `sorted(figuras, key=lambda f: f.area())`.

In [ ]:
# ── Tu turno: implementá la jerarquía ─────────────────────────────────────────

class FiguraGeometrica(ABC):
    @abstractmethod
    def area(self) -> float:
        """Devuelve el area de la figura."""
        ...

    @abstractmethod
    def perimetro(self) -> float:
        """Devuelve el perimetro de la figura."""
        ...

    def describir(self) -> str:
        # TODO: devolver f"{type(self).__name__}: area=..., perimetro=..."
        raise NotImplementedError("Completá este método")


class Circulo(FiguraGeometrica):
    def __init__(self, radio: float):
        # TODO
        pass

    def area(self) -> float:
        # TODO: pi * r^2
        raise NotImplementedError

    def perimetro(self) -> float:
        # TODO: 2 * pi * r
        raise NotImplementedError


class TrianguloRectangulo(FiguraGeometrica):
    def __init__(self, cateto_a: float, cateto_b: float):
        # TODO
        pass

    def area(self) -> float:
        # TODO: (a * b) / 2
        raise NotImplementedError

    def perimetro(self) -> float:
        # TODO: a + b + hipotenusa
        raise NotImplementedError


# Descomentar para probar:
# print(Circulo(5).describir())
# print(TrianguloRectangulo(3, 4).describir())

### Solución

In [ ]:
class FiguraGeometrica(ABC):
    """Clase base abstracta para figuras geométricas planas."""

    @abstractmethod
    def area(self) -> float:
        """Devuelve el area de la figura."""

    @abstractmethod
    def perimetro(self) -> float:
        """Devuelve el perimetro de la figura."""

    def describir(self) -> str:
        """Descripcion textual: nombre, area y perimetro."""
        return (f"{type(self).__name__}: "
                f"area={self.area():.4f}, perimetro={self.perimetro():.4f}")


class Circulo(FiguraGeometrica):
    def __init__(self, radio: float):
        if radio <= 0: raise ValueError("Radio debe ser positivo")
        self._radio = float(radio)

    @property
    def radio(self) -> float: return self._radio

    def area(self) -> float:      return math.pi * self._radio ** 2
    def perimetro(self) -> float: return 2 * math.pi * self._radio
    def __repr__(self):           return f"Circulo(radio={self._radio})"


class TrianguloRectangulo(FiguraGeometrica):
    def __init__(self, cateto_a: float, cateto_b: float):
        if cateto_a <= 0 or cateto_b <= 0: raise ValueError("Catetos deben ser positivos")
        self._a = float(cateto_a)
        self._b = float(cateto_b)

    @property
    def hipotenusa(self) -> float: return math.sqrt(self._a**2 + self._b**2)

    def area(self) -> float:      return (self._a * self._b) / 2
    def perimetro(self) -> float: return self._a + self._b + self.hipotenusa
    def __repr__(self):           return f"TrianguloRectangulo(a={self._a}, b={self._b})"


# ── Demo ──────────────────────────────────────────────────────────────────────
c  = Circulo(5)
t  = TrianguloRectangulo(3, 4)

print(c.describir())   # Circulo: area=78.5398, perimetro=31.4159
print(t.describir())   # TrianguloRectangulo: area=6.0000, perimetro=12.0000
print(f"hipotenusa 3-4-5: {t.hipotenusa}")  # 5.0

# Polimorfismo: función que opera sobre cualquier FiguraGeometrica
figuras = [Circulo(1), TrianguloRectangulo(3, 4), Circulo(3), TrianguloRectangulo(5, 12)]
print("\nFiguras ordenadas por area:")
for f in sorted(figuras, key=lambda f: f.area()):
    print(f"  {f.describir()}")

# El ABC impide instanciar directamente
print("\nIntentando instanciar FiguraGeometrica directamente:")
try:
    FiguraGeometrica()
except TypeError as e:
    print(f"  TypeError: {e}")

## Ejercicio 3 — Jerarquía `Progresion` (§2.4 del libro) 🟡

**Concepto:** Template Method + protocolo iterador.

Reimplementá desde cero la jerarquía de progresiones del libro (§2.4):

```
Progresion
├── _avanzar()          ← hook (sobreescrito por subclases)
├── __iter__()          ← concreto: devuelve self
└── __next__()          ← concreto: llama _avanzar() y devuelve self._actual

ProgresionAritmetica(Progresion)   — suma incremento en cada paso
ProgresionGeometrica(Progresion)   — multiplica base en cada paso
ProgresionFibonacci(Progresion)    — suma los dos anteriores
```

**Reglas:**
- `Progresion.__init__(inicio=0)` almacena el valor en `self._actual`.
- `__next__` debe lanzar `StopIteration` cuando `self._actual is None`.
- El número de términos se controla desde afuera (no dentro de la clase).
- `_avanzar()` en `Progresion` incrementa de a 1 (progresión natural como default).

**Demo esperado:**
```
Aritmética  (inicio=0, delta=5, n=6): 0 5 10 15 20 25
Geométrica  (inicio=1, base=2, n=6):  1 2 4 8 16 32
Fibonacci   (n=8):                    0 1 1 2 3 5 8 13
```

In [ ]:
class Progresion:
    """Clase base para progresiones numéricas (iterator protocol)."""

    def __init__(self, inicio=0):
        self._actual = inicio

    def _avanzar(self):
        """Actualiza self._actual al siguiente valor. Sobreescribir en subclases."""
        self._actual += 1

    def __next__(self):
        if self._actual is None:
            raise StopIteration
        respuesta = self._actual
        self._avanzar()
        return respuesta

    def __iter__(self):
        return self

    def primeros(self, n: int) -> list:
        """Devuelve una lista con los primeros n términos."""
        return [next(self) for _ in range(n)]


class ProgresionAritmetica(Progresion):
    """Suma un incremento constante en cada paso."""

    def __init__(self, inicio=0, incremento=1):
        super().__init__(inicio)
        self._incremento = incremento

    def _avanzar(self):
        self._actual += self._incremento


class ProgresionGeometrica(Progresion):
    """Multiplica por una base constante en cada paso."""

    def __init__(self, inicio=1, base=2.0):
        super().__init__(inicio)
        self._base = base

    def _avanzar(self):
        self._actual *= self._base


class ProgresionFibonacci(Progresion):
    """Cada término = suma de los dos anteriores. F(0)=0, F(1)=1."""

    def __init__(self, inicio=0, segundo=1):
        super().__init__(inicio)
        self._previo = segundo - inicio   # ajuste para que el primer _avanzar sea correcto

    def _avanzar(self):
        self._actual, self._previo = self._actual + self._previo, self._actual


# ── Demo ──────────────────────────────────────────────────────────────────────
pa = ProgresionAritmetica(inicio=0, incremento=5)
pg = ProgresionGeometrica(inicio=1, base=2)
pf = ProgresionFibonacci()

print("Aritmética  (inicio=0, delta=5, n=6):", pa.primeros(6))
print("Geométrica  (inicio=1, base=2, n=6): ", pg.primeros(6))
print("Fibonacci   (n=8):                   ", pf.primeros(8))

# Iteración directa con for
print("\nPrimeros 5 de ProgresionAritmetica(10, 3):")
for v in ProgresionAritmetica(10, 3).primeros(5):
    print(f"  {v}", end="")
print()

# Tests rápidos para Progresion
class TestProgresiones(unittest.TestCase):
    def test_aritmetica(self):
        self.assertEqual(ProgresionAritmetica(0, 3).primeros(4), [0, 3, 6, 9])
    def test_geometrica(self):
        self.assertEqual(ProgresionGeometrica(1, 2).primeros(5), [1, 2, 4, 8, 16])
    def test_fibonacci(self):
        self.assertEqual(ProgresionFibonacci().primeros(8), [0, 1, 1, 2, 3, 5, 8, 13])
    def test_progresion_base(self):
        self.assertEqual(Progresion(10).primeros(4), [10, 11, 12, 13])

res = unittest.TextTestRunner(verbosity=2).run(
    unittest.TestLoader().loadTestsFromTestCase(TestProgresiones))
print(f"\n{'OK' if res.wasSuccessful() else 'FAILED'} — {res.testsRun} tests")

## Ejercicio 4 — `RangoPersonalizado(Secuencia)` 🟡

**Concepto:** Implementar un ADT heredando métodos mixin de una ABC.

La clase `Secuencia` (definida en el setup) es análoga a la del §2.7 del libro:
- Declara abstractos: `__len__` y `__getitem__`.
- Provee como mixin: `__contains__`, `__iter__`, `index`, `count`.

Implementá `RangoPersonalizado(Secuencia)` que represente un rango de enteros
`[inicio, fin)` con un paso determinado — similar a `range()` — pero sin almacenar
los valores en memoria.

**Requisitos:**
- Solo debés implementar `__len__` y `__getitem__`.
- Los métodos `__contains__`, `__iter__`, `index` y `count` deben funcionar **heredados**.
- `__getitem__` debe soportar índices negativos.
- El espacio en memoria debe ser **O(1)** — sin listas internas.

**Demo esperado:**
```python
r = RangoPersonalizado(0, 20, 3)   # 0, 3, 6, 9, 12, 15, 18
len(r)         # 7
r[2]           # 6
r[-1]          # 18
9 in r         # True
7 in r         # False
r.index(12)    # 4
r.count(6)     # 1
list(r)        # [0, 3, 6, 9, 12, 15, 18]
```

In [ ]:
class RangoPersonalizado(Secuencia):
    """Rango de enteros [inicio, fin) con paso, sin almacenar los valores."""

    def __init__(self, inicio: int = 0, fin: int = 0, paso: int = 1):
        if paso == 0: raise ValueError("El paso no puede ser cero")
        self._inicio = inicio
        self._fin    = fin
        self._paso   = paso

    def __len__(self) -> int:
        if self._paso > 0:
            return max(0, (self._fin - self._inicio + self._paso - 1) // self._paso)
        else:
            return max(0, (self._inicio - self._fin - self._paso - 1) // (-self._paso))

    def __getitem__(self, j: int):
        if j < 0:
            j += len(self)              # soporte de indices negativos
        if not (0 <= j < len(self)):
            raise IndexError("indice fuera de rango")
        return self._inicio + j * self._paso

    def __repr__(self) -> str:
        return f"RangoPersonalizado({self._inicio}, {self._fin}, {self._paso})"


# ── Demo ──────────────────────────────────────────────────────────────────────
r = RangoPersonalizado(0, 20, 3)    # 0, 3, 6, 9, 12, 15, 18
print(f"len(r)     = {len(r)}")           # 7
print(f"r[2]       = {r[2]}")             # 6
print(f"r[-1]      = {r[-1]}")            # 18
print(f"9 in r     = {9 in r}")           # True  — usa __contains__ heredado
print(f"7 in r     = {7 in r}")           # False
print(f"r.index(12)= {r.index(12)}")      # 4     — usa index() heredado
print(f"r.count(6) = {r.count(6)}")       # 1     — usa count() heredado
print(f"list(r)    = {list(r)}")          # usa __iter__ heredado

# Tests
class TestRangoPersonalizado(unittest.TestCase):
    def setUp(self):
        self.r = RangoPersonalizado(0, 20, 3)

    def test_len(self):
        self.assertEqual(len(self.r), 7)
    def test_getitem(self):
        self.assertEqual(self.r[2], 6)
    def test_negativo(self):
        self.assertEqual(self.r[-1], 18)
    def test_contains_heredado(self):
        self.assertIn(9, self.r)
        self.assertNotIn(7, self.r)
    def test_iter_heredado(self):
        self.assertEqual(list(self.r), [0, 3, 6, 9, 12, 15, 18])
    def test_index_heredado(self):
        self.assertEqual(self.r.index(12), 4)
    def test_count_heredado(self):
        self.assertEqual(self.r.count(6), 1)

res = unittest.TextTestRunner(verbosity=2).run(
    unittest.TestLoader().loadTestsFromTestCase(TestRangoPersonalizado))
print(f"\n{'OK' if res.wasSuccessful() else 'FAILED'} — {res.testsRun} tests")

## Desafío — `SecuenciaOrdenada(Secuencia)` 🔴🔴

**Concepto:** ADT Secuencia ordenada con búsqueda eficiente.

Implementá `SecuenciaOrdenada` que:

1. Hereda de `Secuencia` — solo implementa `__len__` y `__getitem__`.
2. Mantiene los datos **siempre ordenados** internamente.
3. Agrega `insertar(val)` que usa `bisect.insort` para insertar en O(n) manteniendo el orden.
4. Agrega `buscar(val)` que usa `bisect.bisect_left` para búsqueda binaria O(log n).
5. Hereda automáticamente: `__contains__`, `__iter__`, `index`, `count`.

**¿Por qué bisect?**

| Operación | Con lista normal | Con bisect |
|-----------|-----------------|------------|
| Buscar posición de inserción | O(n) | **O(log n)** |
| Verificar si existe | O(n) via `in` | **O(log n)** vía `buscar` |
| Insertar (desplazar) | O(n) | O(n) — igual, pero posición en O(log n) |

In [ ]:
import bisect

# ── Tu turno: completá la implementación ──────────────────────────────────────

class SecuenciaOrdenada(Secuencia):
    """Secuencia que mantiene sus elementos ordenados en todo momento."""

    def __init__(self, iterable=None):
        self._datos = sorted(iterable) if iterable else []

    def __len__(self) -> int:
        # TODO: return len(self._datos)
        raise NotImplementedError

    def __getitem__(self, j: int):
        # TODO: return self._datos[j]
        raise NotImplementedError

    def insertar(self, val) -> None:
        """Inserta val manteniendo el orden. O(log n) para hallar posicion + O(n) para desplazar."""
        # TODO: bisect.insort(self._datos, val)
        raise NotImplementedError

    def buscar(self, val) -> bool:
        """Devuelve True si val esta en la secuencia, usando bisect. O(log n)."""
        # TODO: usar bisect.bisect_left y comparar
        raise NotImplementedError


# Descomentar para probar:
# s = SecuenciaOrdenada([5, 1, 3, 2, 4])
# print(list(s))       # [1, 2, 3, 4, 5]
# s.insertar(0)
# print(list(s))       # [0, 1, 2, 3, 4, 5]
# print(s.buscar(3))   # True

### Solución

In [ ]:
import bisect

class SecuenciaOrdenada(Secuencia):
    """Secuencia que mantiene sus elementos ordenados en todo momento."""

    def __init__(self, iterable=None):
        self._datos = sorted(iterable) if iterable else []

    def __len__(self) -> int:
        return len(self._datos)

    def __getitem__(self, j: int):
        return self._datos[j]         # soporta indices negativos automáticamente

    def insertar(self, val) -> None:
        """Inserta val manteniendo el orden usando bisect — O(log n) + O(n)."""
        bisect.insort(self._datos, val)

    def buscar(self, val) -> bool:
        """Busqueda binaria — O(log n)."""
        pos = bisect.bisect_left(self._datos, val)
        return pos < len(self._datos) and self._datos[pos] == val

    def __repr__(self) -> str:
        return f"SecuenciaOrdenada({self._datos!r})"


# ── Demo ──────────────────────────────────────────────────────────────────────
s = SecuenciaOrdenada([5, 1, 9, 2, 7, 3])
print(f"Inicial:          {list(s)}")       # [1, 2, 3, 5, 7, 9]
print(f"len:              {len(s)}")         # 6
print(f"s[0]:             {s[0]}")           # 1
print(f"s[-1]:            {s[-1]}")          # 9

s.insertar(4)
s.insertar(0)
s.insertar(10)
print(f"Tras 3 inserts:   {list(s)}")       # [0, 1, 2, 3, 4, 5, 7, 9, 10]

print(f"\nbuscar(5):        {s.buscar(5)}")   # True  — O(log n)
print(f"buscar(6):        {s.buscar(6)}")    # False

# Los metodos heredados siguen funcionando
print(f"\n3 in s (heredado):   {3 in s}")
print(f"s.index(7):          {s.index(7)}")
print(f"s.count(5):          {s.count(5)}")
print(f"list(iter(s)):       {list(iter(s))}")

# Tests
class TestSecuenciaOrdenada(unittest.TestCase):
    def setUp(self):
        self.s = SecuenciaOrdenada([5, 1, 9, 2, 7, 3])

    def test_ordenada_al_crear(self):
        self.assertEqual(list(self.s), [1, 2, 3, 5, 7, 9])
    def test_insertar_mantiene_orden(self):
        self.s.insertar(4)
        self.assertEqual(list(self.s), [1, 2, 3, 4, 5, 7, 9])
    def test_buscar_existente(self):
        self.assertTrue(self.s.buscar(7))
    def test_buscar_inexistente(self):
        self.assertFalse(self.s.buscar(6))
    def test_contains_heredado(self):
        self.assertIn(5, self.s)
    def test_iter_heredado(self):
        self.assertEqual(list(self.s), sorted([5, 1, 9, 2, 7, 3]))
    def test_index_heredado(self):
        self.assertEqual(self.s.index(7), 4)

res = unittest.TextTestRunner(verbosity=2).run(
    unittest.TestLoader().loadTestsFromTestCase(TestSecuenciaOrdenada))
print(f"\n{'OK' if res.wasSuccessful() else 'FAILED'} — {res.testsRun} tests")

## 🧪 Suite Completa de Tests

In [ ]:
class TestFiguraGeometrica(unittest.TestCase):
    def test_circulo_area(self):
        c = Circulo(1)
        self.assertAlmostEqual(c.area(), math.pi, places=10)

    def test_circulo_perimetro(self):
        c = Circulo(1)
        self.assertAlmostEqual(c.perimetro(), 2 * math.pi, places=10)

    def test_triangulo_area(self):
        t = TrianguloRectangulo(3, 4)
        self.assertAlmostEqual(t.area(), 6.0)

    def test_triangulo_hipotenusa(self):
        t = TrianguloRectangulo(3, 4)
        self.assertAlmostEqual(t.hipotenusa, 5.0)

    def test_no_instanciar_abc(self):
        with self.assertRaises(TypeError):
            FiguraGeometrica()

    def test_describir_hereda(self):
        desc = Circulo(5).describir()
        self.assertIn("Circulo", desc)
        self.assertIn("area", desc)


# ── Recopilar y correr todas las suites ───────────────────────────────────────
suites = [
    unittest.TestLoader().loadTestsFromTestCase(cls)
    for cls in [
        TestCuadrado,
        TestFiguraGeometrica,
        TestProgresiones,
        TestRangoPersonalizado,
        TestSecuenciaOrdenada,
    ]
]
all_tests = unittest.TestSuite(suites)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(all_tests)

print("\n" + "="*60)
total = result.testsRun
ok    = total - len(result.failures) - len(result.errors)
print(f"RESULTADO FINAL: {ok}/{total} tests pasaron")
if result.wasSuccessful():
    print("Todos los ejercicios implementados correctamente.")
else:
    print("Revisá los errores arriba.")

## Síntesis: patrones aplicados

| Ejercicio | Patrón principal | Conceptos clave |
|-----------|-----------------|-----------------|
| Ex 1 — `Cuadrado(Rectangulo)` | Herencia simple | `super()`, sustitución de Liskov, override parcial |
| Ex 2 — `FiguraGeometrica(ABC)` | Abstract Base Class | `@abstractmethod`, polimorfismo, contrato de interfaz |
| Ex 3 — Jerarquía `Progresion` | Template Method | `_avanzar()` hook, protocolo iterador, `__iter__`/`__next__` |
| Ex 4 — `RangoPersonalizado(Secuencia)` | Mixin ABC | Mínima implementación obligatoria, comportamiento heredado gratis |
| Desafío — `SecuenciaOrdenada` | ADT con invariante | `bisect`, O(log n) búsqueda, invariante de orden mantenida |

### Comparación diseño con ABC vs sin ABC

| Aspecto | Sin ABC | Con ABC |
|---------|---------|---------|
| Error al olvidar un método | En tiempo de ejecución | **En tiempo de instanciación** |
| Documentación de interfaz | Implícita | Explícita (lectura del ABC) |
| Métodos mixin | Manual en cada clase | **Automáticos si ABC los provee** |
| `isinstance` check | Solo por herencia real | También vía `register()` |

### Invariante de `SecuenciaOrdenada`

Un **invariante** es una propiedad que debe ser cierta antes y después de cada operación:

```
∀ i < j: s[i] <= s[j]    (la secuencia está ordenada en todo momento)
```

`insertar()` preserva este invariante usando `bisect.insort` — el invariante es el
valor agregado que la clase ofrece sobre una lista simple.

## Preguntas de Reflexión

Respondé brevemente (2–4 oraciones) antes de la próxima clase:

1. **Sustitución de Liskov:** `Cuadrado` hereda de `Rectangulo`. ¿Es esta herencia realmente correcta desde el punto de vista del principio LSP? ¿Qué pasa si alguien hace `r.alto = 5` sin tocar `r.ancho` en un `Rectangulo`? ¿Podría hacerse en un `Cuadrado`?

2. **Mínimo abstracto:** En `RangoPersonalizado`, solo implementaste `__len__` y `__getitem__`. ¿Por qué eso es suficiente para que `__contains__`, `__iter__`, `index` y `count` funcionen sin redefinirlos?

3. **Template Method:** La clase `Progresion` sigue el patrón Template Method. Identificá el "template" (método invariante) y el "hook" (método variable). ¿Qué garantiza que el template funcione para todas las subclases?

4. **Invariante de clase:** `SecuenciaOrdenada` mantiene el invariante de orden. ¿Qué pasaría si expusieras `_datos` directamente y alguien hiciera `s._datos.append(0)`? ¿Cómo protegerías mejor el invariante?

5. **duck typing vs ABC:** `FiguraGeometrica` es un ABC. Podrías haber implementado lo mismo con duck typing (sin herencia, solo implementando `area()` y `perimetro()`). ¿Cuáles son las ventajas y desventajas de cada enfoque en un equipo grande?

---

**Ejercicios Clase 2 — Polimorfismo y ADTs**  
*Estructuras de Datos y Algoritmos en Python*  
Goodrich, Tamassia & Goldwasser — Capítulo 2

Material didáctico bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).  
Código de ejemplo con fines educativos.